In [1]:
# using PCA
"""
Feature Preparation — VGG-16 Block 4 (Upsampled Data): Split, Scale, PCA
--------------------------------------------------------------------------
Run this once per upstream change (new upsampled features, a different
test_size/random_state, or a different PCA variance threshold). It loads the
raw 512-d VGG-16 block4 features, does the train/test split, fits the scaler
and PCA on the training data only (leakage-free), and saves everything the
training script needs — so you can iterate on RandomForest hyperparameters
in a separate script without repeating this fairly expensive step every time.

Outputs (all to OUTPUT_DIR):
  X_train_pca_block4_upsampled.npy, X_test_pca_block4_upsampled.npy
  y_train_block4_upsampled.npy,     y_test_block4_upsampled.npy
  scaler_block4_upsampled.pkl,      pca_block4_upsampled.pkl
  pca_meta_block4_upsampled.json    (config + resulting stats, read by the
                                      training script so it can re-log them)
"""

import json
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
import gc


# ── Settings ───────────────────────────────────────────────────────────────────

FEATURES_FILE = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/upsampled_features_block4pool.npy"
LABELS_FILE   = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/upsampled_labels_block4pool.npy"
OUTPUT_DIR    = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT"


# ── Config ─────────────────────────────────────────────────────────────────────

PREP_CONFIG = {
    "vgg16_block":            4,
    "feature_dim":            512,
    "test_size":               0.20,
    "random_state":            42,
    "pca_variance_threshold": 0.95,   # keep enough components to explain 95% of
                                       # variance; lower = more compression, more
                                       # risk of losing useful signal
}


# ── Step 1: Load data ──────────────────────────────────────────────────────────

print("Loading features and labels...")

features = np.load(FEATURES_FILE)   # shape: (N, 512)
labels   = np.load(LABELS_FILE)

# Downcast now — CNN feature extractors output float32 anyway, so this loses
# no real precision, and halves the memory footprint of everything downstream.
features = features.astype(np.float32, copy=False)
labels   = labels.astype(np.int32, copy=False)

print(f"Features shape : {features.shape}  ({features.nbytes / 1e9:.2f} GB, dtype={features.dtype})")
print(f"Labels shape   : {labels.shape}")


# ── Step 2: Train / test split ────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size    = PREP_CONFIG["test_size"],
    random_state = PREP_CONFIG["random_state"],
    stratify     = labels
)

print(f"Train set : {X_train.shape[0]} images")
print(f"Test set  : {X_test.shape[0]} images")

# train_test_split allocated fresh train/test arrays — the originals are now
# pure overhead sitting in RAM.
del features, labels
gc.collect()


# ── Step 3: Scale features ─────────────────────────────────────────────────────
# Fit on train only, then apply those same stats to test — keeps the test set
# leakage-free regardless of how the classifier handles class balance.

print("\nScaling features...")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = scaler.transform(X_test).astype(np.float32, copy=False)
gc.collect()


# ── Step 4: Reduce dimensionality (PCA) ───────────────────────────────────────
# Fit on train only (same leakage-free rule as the scaler above): components
# are learned from training data, then applied as-is to test.

print(f"\nReducing dimensionality with PCA (target: "
      f"{PREP_CONFIG['pca_variance_threshold']*100:.0f}% variance retained)...")

pca     = PCA(n_components=PREP_CONFIG["pca_variance_threshold"], svd_solver="full")
X_train = pca.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = pca.transform(X_test).astype(np.float32, copy=False)
gc.collect()

print(f"PCA kept {pca.n_components_} components (down from {PREP_CONFIG['feature_dim']}), "
      f"explained variance: {pca.explained_variance_ratio_.sum():.4f}")


# ── Step 5: Save everything the training script will need ────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(f"{OUTPUT_DIR}/X_train_pca_block4_upsampled.npy", X_train)
np.save(f"{OUTPUT_DIR}/X_test_pca_block4_upsampled.npy",  X_test)
np.save(f"{OUTPUT_DIR}/y_train_block4_upsampled.npy",     y_train)
np.save(f"{OUTPUT_DIR}/y_test_block4_upsampled.npy",      y_test)

joblib.dump(scaler, f"{OUTPUT_DIR}/scaler_block4_upsampled.pkl")
joblib.dump(pca,    f"{OUTPUT_DIR}/pca_block4_upsampled.pkl")

meta = {
    **PREP_CONFIG,
    "pca_n_components":       int(pca.n_components_),
    "pca_explained_variance":  float(pca.explained_variance_ratio_.sum()),
    "n_train_samples":        int(X_train.shape[0]),
    "n_test_samples":         int(X_test.shape[0]),
}
with open(f"{OUTPUT_DIR}/pca_meta_block4_upsampled.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved prepared arrays, scaler, PCA transformer, and metadata to {OUTPUT_DIR}/")
print("You can now run the training script as many times as you like — it'll "
      "reuse this without redoing the split/scale/PCA steps.")

Loading features and labels...
Features shape : (125184, 512)  (0.26 GB, dtype=float32)
Labels shape   : (125184,)
Train set : 100147 images
Test set  : 25037 images

Scaling features...

Reducing dimensionality with PCA (target: 95% variance retained)...
PCA kept 320 components (down from 512), explained variance: 0.9503

Saved prepared arrays, scaler, PCA transformer, and metadata to C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/
You can now run the training script as many times as you like — it'll reuse this without redoing the split/scale/PCA steps.


In [2]:
# Make PCA more strict

"""
Feature Preparation — VGG-16 Block 4 (Upsampled Data): Split, Scale, PCA
--------------------------------------------------------------------------
Run this once per upstream change (new upsampled features, a different
test_size/random_state, or a different PCA variance threshold). It loads the
raw 512-d VGG-16 block4 features, does the train/test split, fits the scaler
and PCA on the training data only (leakage-free), and saves everything the
training script needs — so you can iterate on RandomForest hyperparameters
in a separate script without repeating this fairly expensive step every time.

Outputs (all to OUTPUT_DIR):
  X_train_pca_block4_upsampled.npy, X_test_pca_block4_upsampled.npy
  y_train_block4_upsampled.npy,     y_test_block4_upsampled.npy
  scaler_block4_upsampled.pkl,      pca_block4_upsampled.pkl
  pca_meta_block4_upsampled.json    (config + resulting stats, read by the
                                     training script so it can re-log them)
"""

import json
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
import gc


# ── Settings ───────────────────────────────────────────────────────────────────

FEATURES_FILE = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/upsampled_features_block4pool.npy"
LABELS_FILE   = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/upsampled_labels_block4pool.npy"
OUTPUT_DIR    = "C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT"


# ── Config ─────────────────────────────────────────────────────────────────────

PREP_CONFIG = {
    "vgg16_block":            4,
    "feature_dim":            512,
    "test_size":              0.20,
    "random_state":           42,
    "pca_n_components":       64,  # STRICT HARSH LIMIT: Guarantees a tiny memory footprint.
                                   # Adjust to 32 (harsher) or 128 (softer) as RAM permits.
}


# ── Step 1: Load data ──────────────────────────────────────────────────────────

print("Loading features and labels...")

features = np.load(FEATURES_FILE)   # shape: (N, 512)
labels   = np.load(LABELS_FILE)

# Downcast now — CNN feature extractors output float32 anyway, so this loses
# no real precision, and halves the memory footprint of everything downstream.
features = features.astype(np.float32, copy=False)
labels   = labels.astype(np.int32, copy=False)

print(f"Features shape : {features.shape}  ({features.nbytes / 1e9:.2f} GB, dtype={features.dtype})")
print(f"Labels shape   : {labels.shape}")


# ── Step 2: Train / test split ────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size    = PREP_CONFIG["test_size"],
    random_state = PREP_CONFIG["random_state"],
    stratify     = labels
)

print(f"Train set : {X_train.shape[0]} images")
print(f"Test set  : {X_test.shape[0]} images")

# train_test_split allocated fresh train/test arrays — the originals are now
# pure overhead sitting in RAM.
del features, labels
gc.collect()


# ── Step 3: Scale features ─────────────────────────────────────────────────────
# Fit on train only, then apply those same stats to test — keeps the test set
# leakage-free regardless of how the classifier handles class balance.

print("\nScaling features...")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = scaler.transform(X_test).astype(np.float32, copy=False)
gc.collect()


# ── Step 4: Reduce dimensionality (PCA) ───────────────────────────────────────
# Fit on train only (same leakage-free rule as the scaler above): components
# are learned from training data, then applied as-is to test.

print(f"\nReducing dimensionality with PCA (strict limit: "
      f"{PREP_CONFIG['pca_n_components']} components)...")

# 'randomized' solver is significantly less memory intensive during the fit
# process compared to 'full', which is vital for upsampled datasets.
pca     = PCA(
    n_components=PREP_CONFIG["pca_n_components"], 
    svd_solver="randomized", 
    random_state=PREP_CONFIG["random_state"]
)

X_train = pca.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = pca.transform(X_test).astype(np.float32, copy=False)
gc.collect()

print(f"PCA kept {pca.n_components_} components (down from {PREP_CONFIG['feature_dim']}), "
      f"explained variance: {pca.explained_variance_ratio_.sum():.4f}")


# ── Step 5: Save everything the training script will need ────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(f"{OUTPUT_DIR}/X_train_pca_block4_upsampled_2.npy", X_train)
np.save(f"{OUTPUT_DIR}/X_test_pca_block4_upsampled_2.npy",  X_test)
np.save(f"{OUTPUT_DIR}/y_train_block4_upsampled_2.npy",     y_train)
np.save(f"{OUTPUT_DIR}/y_test_block4_upsampled_2.npy",      y_test)

joblib.dump(scaler, f"{OUTPUT_DIR}/scaler_block4_upsampled_2.pkl")
joblib.dump(pca,    f"{OUTPUT_DIR}/pca_block4_upsampled_2.pkl")

meta = {
    **PREP_CONFIG,
    "pca_n_components":       int(pca.n_components_),
    "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
    "n_train_samples":        int(X_train.shape[0]),
    "n_test_samples":         int(X_test.shape[0]),
}
with open(f"{OUTPUT_DIR}/pca_meta_block4_upsampled.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved prepared arrays, scaler, PCA transformer, and metadata to {OUTPUT_DIR}/")
print("You can now run the training script as many times as you like — it'll "
      "reuse this without redoing the split/scale/PCA steps.")

Loading features and labels...
Features shape : (125184, 512)  (0.26 GB, dtype=float32)
Labels shape   : (125184,)
Train set : 100147 images
Test set  : 25037 images

Scaling features...

Reducing dimensionality with PCA (strict limit: 64 components)...
PCA kept 64 components (down from 512), explained variance: 0.7120

Saved prepared arrays, scaler, PCA transformer, and metadata to C:/Users/niric/Desktop/classes/AI and ML/CLEAN_DATASET_UPSAMPLED/OUTPUT/
You can now run the training script as many times as you like — it'll reuse this without redoing the split/scale/PCA steps.


In [1]:
# Make PCA more strict - again after labeling mistake

"""
Feature Preparation — VGG-16 Block 4 (Upsampled Data): Split, Scale, PCA
--------------------------------------------------------------------------
Run this once per upstream change (new upsampled features, a different
test_size/random_state, or a different PCA variance threshold). It loads the
raw 512-d VGG-16 block4 features, does the train/test split, fits the scaler
and PCA on the training data only (leakage-free), and saves everything the
training script needs — so you can iterate on RandomForest hyperparameters
in a separate script without repeating this fairly expensive step every time.

Outputs (all to OUTPUT_DIR):
  X_train_pca_block4_upsampled.npy, X_test_pca_block4_upsampled.npy
  y_train_block4_upsampled.npy,     y_test_block4_upsampled.npy
  scaler_block4_upsampled.pkl,      pca_block4_upsampled.pkl
  pca_meta_block4_upsampled.json    (config + resulting stats, read by the
                                     training script so it can re-log them)
"""

import json
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
import gc


# ── Settings ───────────────────────────────────────────────────────────────────

FEATURES_FILE = "C:/Users/niric/Desktop/classes/AI and ML/UPSAMPLED_OUTPUT/upsampled_features_block4pool_3.npy"
LABELS_FILE   = "C:/Users/niric/Desktop/classes/AI and ML/UPSAMPLED_OUTPUT/upsampled_labels_block4pool_3.npy"
OUTPUT_DIR    = "C:/Users/niric/Desktop/classes/AI and ML/UPSAMPLED_OUTPUT/"


# ── Config ─────────────────────────────────────────────────────────────────────

PREP_CONFIG = {
    "vgg16_block":            4,
    "feature_dim":            512,
    "test_size":              0.20,
    "random_state":           42,
    "pca_n_components":       64,  # STRICT HARSH LIMIT: Guarantees a tiny memory footprint.
                                   # Adjust to 32 (harsher) or 128 (softer) as RAM permits.
}


# ── Step 1: Load data ──────────────────────────────────────────────────────────

print("Loading features and labels...")

features = np.load(FEATURES_FILE)   # shape: (N, 512)
labels   = np.load(LABELS_FILE)

# Downcast now — CNN feature extractors output float32 anyway, so this loses
# no real precision, and halves the memory footprint of everything downstream.
features = features.astype(np.float32, copy=False)
labels   = labels.astype(np.int32, copy=False)

print(f"Features shape : {features.shape}  ({features.nbytes / 1e9:.2f} GB, dtype={features.dtype})")
print(f"Labels shape   : {labels.shape}")


# ── Step 2: Train / test split ────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size    = PREP_CONFIG["test_size"],
    random_state = PREP_CONFIG["random_state"],
    stratify     = labels
)

print(f"Train set : {X_train.shape[0]} images")
print(f"Test set  : {X_test.shape[0]} images")

# train_test_split allocated fresh train/test arrays — the originals are now
# pure overhead sitting in RAM.
del features, labels
gc.collect()


# ── Step 3: Scale features ─────────────────────────────────────────────────────
# Fit on train only, then apply those same stats to test — keeps the test set
# leakage-free regardless of how the classifier handles class balance.

print("\nScaling features...")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = scaler.transform(X_test).astype(np.float32, copy=False)
gc.collect()


# ── Step 4: Reduce dimensionality (PCA) ───────────────────────────────────────
# Fit on train only (same leakage-free rule as the scaler above): components
# are learned from training data, then applied as-is to test.

print(f"\nReducing dimensionality with PCA (strict limit: "
      f"{PREP_CONFIG['pca_n_components']} components)...")

# 'randomized' solver is significantly less memory intensive during the fit
# process compared to 'full', which is vital for upsampled datasets.
pca     = PCA(
    n_components=PREP_CONFIG["pca_n_components"], 
    svd_solver="randomized", 
    random_state=PREP_CONFIG["random_state"]
)

X_train = pca.fit_transform(X_train).astype(np.float32, copy=False)
X_test  = pca.transform(X_test).astype(np.float32, copy=False)
gc.collect()

print(f"PCA kept {pca.n_components_} components (down from {PREP_CONFIG['feature_dim']}), "
      f"explained variance: {pca.explained_variance_ratio_.sum():.4f}")


# ── Step 5: Save everything the training script will need ────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(f"{OUTPUT_DIR}/X_train_pca_block4_upsampled_2.npy", X_train)
np.save(f"{OUTPUT_DIR}/X_test_pca_block4_upsampled_2.npy",  X_test)
np.save(f"{OUTPUT_DIR}/y_train_block4_upsampled_2.npy",     y_train)
np.save(f"{OUTPUT_DIR}/y_test_block4_upsampled_2.npy",      y_test)

joblib.dump(scaler, f"{OUTPUT_DIR}/scaler_block4_upsampled_2.pkl")
joblib.dump(pca,    f"{OUTPUT_DIR}/pca_block4_upsampled_2.pkl")

meta = {
    **PREP_CONFIG,
    "pca_n_components":       int(pca.n_components_),
    "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
    "n_train_samples":        int(X_train.shape[0]),
    "n_test_samples":         int(X_test.shape[0]),
}
with open(f"{OUTPUT_DIR}/pca_meta_block4_upsampled.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nSaved prepared arrays, scaler, PCA transformer, and metadata to {OUTPUT_DIR}/")
print("You can now run the training script as many times as you like — it'll "
      "reuse this without redoing the split/scale/PCA steps.")

Loading features and labels...
Features shape : (125184, 512)  (0.26 GB, dtype=float32)
Labels shape   : (125184,)
Train set : 100147 images
Test set  : 25037 images

Scaling features...

Reducing dimensionality with PCA (strict limit: 64 components)...
PCA kept 64 components (down from 512), explained variance: 0.7120

Saved prepared arrays, scaler, PCA transformer, and metadata to C:/Users/niric/Desktop/classes/AI and ML/UPSAMPLED_OUTPUT//
You can now run the training script as many times as you like — it'll reuse this without redoing the split/scale/PCA steps.
